# Fine-Tune Whisper Medium for Amharic ASR
**Dataset**: Leyu Amharic (4 dialects, ~27k samples) | **Model**: `openai/whisper-medium`

> **Before running**: Runtime → Change runtime type → **T4 GPU**

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate jiwer librosa soundfile accelerate huggingface_hub

## Step 2: Authenticate with HuggingFace Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 3: Configuration — Update Before Running

In [ ]:
# --- SMOKE TEST MODE ---
# Set True for a fast end-to-end validation run (~10 min).
# Proves the pipeline works BEFORE committing to a 3-5h full run.
# Set False for the real training.
SMOKE_TEST = True

HF_USERNAME = "hundehanna"
MODEL_NAME = f"{HF_USERNAME}/whisper-medium-amharic" + ("-smoke" if SMOKE_TEST else "")

# Leyu dialect dataset IDs on HuggingFace Hub
DATASET_NAMES = [
    "leyu-amharic/leyu-amharic-gojjam-dialect",   # 10.6k rows, 81.6h
    "leyu-amharic/leyu-amharic-gonder-dialect",    # 8.99k rows
    "leyu-amharic/leyu-amharic-wello-dialect",     # 4.86k rows
    "leyu-amharic/leyu-amharic-shewa-dialect",     # 2.59k rows
]

AUDIO_COLUMN = "audio"
TRANSCRIPT_COLUMN = "text"
SAMPLE_RATE = 16000
MIN_DURATION = 1.0   # seconds
MAX_DURATION = 30.0  # Whisper maximum

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL TRAINING'}")
print(f"Target model: {MODEL_NAME}")

## Step 4: Load & Combine All Dialect Datasets

In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets

dialect_splits = [load_dataset(name, split="train") for name in DATASET_NAMES]
combined = concatenate_datasets(dialect_splits)
print(f"Total samples (all dialects): {len(combined)}")

if SMOKE_TEST:
    # Tiny subset: only enough samples to validate the pipeline end-to-end.
    combined = combined.shuffle(seed=42).select(range(500))
    print(f"SMOKE TEST: downsampled to {len(combined)} samples")

## Step 5: Resample Audio & Filter by Duration

In [ ]:
combined = combined.cast_column(AUDIO_COLUMN, Audio(sampling_rate=SAMPLE_RATE))

def is_valid_duration(example):
    duration = len(example[AUDIO_COLUMN]["array"]) / example[AUDIO_COLUMN]["sampling_rate"]
    return MIN_DURATION <= duration <= MAX_DURATION

combined = combined.filter(is_valid_duration)
print(f"After duration filter: {len(combined)} samples")

## Step 6: Create Train / Validation / Test Splits
> The Leyu datasets only ship with a 'train' split — we create our own 80/10/10 split.

In [ ]:
train_testval = combined.train_test_split(test_size=0.2, seed=42)
test_val = train_testval["test"].train_test_split(test_size=0.5, seed=42)

train_ds = train_testval["train"]
val_ds = test_val["train"]
test_ds = test_val["test"]

print(f"Train:      {len(train_ds):,}")
print(f"Validation: {len(val_ds):,}")
print(f"Test:       {len(test_ds):,}")

## Step 7: Load Processor & Extract Features

In [ ]:
import re
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-medium",
    language="amharic",
    task="transcribe"
)

# Whisper's decoder max length. Labels longer than this crash training.
MAX_LABEL_LEN = 448

def normalize(text):
    return re.sub(r"\s+", " ", text.strip())

def prepare_dataset(example):
    audio = example[AUDIO_COLUMN]
    example["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    example["labels"] = processor.tokenizer(normalize(example[TRANSCRIPT_COLUMN])).input_ids
    return example

cols_to_remove = train_ds.column_names
train_ds = train_ds.map(prepare_dataset, remove_columns=cols_to_remove, num_proc=2)
val_ds   = val_ds.map(prepare_dataset,   remove_columns=cols_to_remove, num_proc=2)
test_ds  = test_ds.map(prepare_dataset,  remove_columns=cols_to_remove, num_proc=2)

# Filter out samples whose tokenized transcript exceeds Whisper's decoder limit.
# Amharic in Ethiopic script tokenizes inefficiently; some transcripts that
# look short produce >448 tokens and would crash training.
def labels_in_range(ex):
    return len(ex["labels"]) <= MAX_LABEL_LEN

before_train, before_val, before_test = len(train_ds), len(val_ds), len(test_ds)
train_ds = train_ds.filter(labels_in_range)
val_ds   = val_ds.filter(labels_in_range)
test_ds  = test_ds.filter(labels_in_range)
print(f"Dropped (train): {before_train - len(train_ds)} | (val): {before_val - len(val_ds)} | (test): {before_test - len(test_ds)}")
print(f"After filter — train: {len(train_ds):,} | val: {len(val_ds):,} | test: {len(test_ds):,}")
print("Feature extraction complete.")

## Step 8: Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## Step 9: WER Metric

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

## Step 10: Load Model

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

## Step 11: Training Arguments

In [ ]:
from transformers import Seq2SeqTrainingArguments

# Smoke-test uses tiny values; full run uses real hyperparameters.
if SMOKE_TEST:
    max_steps, warmup, eval_every, save_every, log_every = 30, 5, 15, 15, 5
else:
    max_steps, warmup, eval_every, save_every, log_every = 4000, 500, 200, 200, 25

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_NAME,
    max_steps=max_steps,
    learning_rate=1e-5,
    warmup_steps=warmup,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",          # was "evaluation_strategy" before transformers 4.46
    eval_steps=eval_every,
    save_steps=save_every,
    logging_steps=log_every,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    push_to_hub=True,
    hub_model_id=MODEL_NAME,
)

## Step 12: Train!

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,  # was "tokenizer" before transformers 4.46
)

trainer.train()

## Step 13: Push Model to HuggingFace Hub

In [ ]:
trainer.push_to_hub()
processor.save_pretrained(training_args.output_dir)
print(f"Model live at: https://huggingface.co/{MODEL_NAME}")

## Step 14: Evaluate on Test Set

In [ ]:
from transformers import pipeline as hf_pipeline

asr = hf_pipeline("automatic-speech-recognition", model=MODEL_NAME, device=0)

predictions, references = [], []
for sample in test_ds.select(range(100)):  # quick eval on 100 samples
    pred = asr(sample["audio"])["text"]
    predictions.append(pred)
    references.append(sample[TRANSCRIPT_COLUMN])

wer = wer_metric.compute(predictions=predictions, references=references)
print(f"Test WER (100 samples): {wer * 100:.2f}%")